In [4]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()


✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [72]:
duck.sql(
    r"""
    select 
        distinct on (source_id, old_zoho_id)
        f.source_id,
        f.name,
        case
            when f.source_id like 'leads_%' then right(f.source_id, -6)
            when f.source_id like '38675%' then f.source_id
            else ez.zoho_id
        end as old_zoho_id,
        regexp_replace(replace(naf_code, '.', ''), '^(\d{2})(\d{2})(\d)$', '\1.\2.\3') as padoa_naf_code,
        regexp_replace(rpad(replace(d."WZ_Code", '.', ''), 5, '0'), '^(\d{2})(\d{2})(\d)$', '\1.\2.\3') as deal_naf_code,
        padoa_naf_code != deal_naf_code and deal_naf_code is not null as diff
    from read_csv('/Users/adrienblanquer/Downloads/naf_code.prod-bas_wellinjob.csv') f
    left join pg.bas_firms.easybill_medisoft em
        on em.id = f.source_id
    left join pg.bas_firms.easybill_zoho ez
        on ez.easybill_id = source_id or em.easybill_id = ez.easybill_id
    left join pg.zoho.Deals d
        on d.Account_Name = ez.zoho_id
    where old_zoho_id is not null
    order by d.Created_Time desc nulls last
    """
).to_csv('naf_code_diffs.csv')

In [21]:
duck.sql("select * from pg.bas_firms.easybill_zoho where easybill_id = '104000020'")

┌───────┬─────────────┬────────────────────┐
│  id   │ easybill_id │      zoho_id       │
│ int32 │   varchar   │      varchar       │
├───────┼─────────────┼────────────────────┤
│ 10221 │ 104000020   │ 386758000012226931 │
└───────┴─────────────┴────────────────────┘